<a href="https://colab.research.google.com/github/msadhi2007-create/ASSIGNMENT/blob/main/streamlitpre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()  # upload archive__3_.zip here

!unzip -o "archive__3_.zip" -d /content

Saving archive (3).zip to archive (3) (1).zip
unzip:  cannot find or open archive__3_.zip, archive__3_.zip.zip or archive__3_.zip.ZIP.


In [16]:
import pandas as pd

df = pd.read_csv("/content/tourism_recommendation_dataset_en.csv")

df = df[[
    "age", "ticket_price", "visit_duration_hours", "spend_amount",
    "other_spend", "is_group_tour", "group_fee", "trip_days", "rating"
]]

df.to_csv("cleaned_tourism_recommendation_dataset.csv", index=False)
print("Saved:", df.shape)

Saved: (100000, 9)


In [26]:
import getpass
ngrok_key = getpass.getpass("Enter your ngrok authtoken: ")

Enter your ngrok authtoken: ··········


In [27]:
!pip install -q -U streamlit pyngrok

In [34]:
%%writefile app.py

import streamlit as st
import pandas as pd

st.set_page_config(
    page_title="Tourist Rating Prediction",
    page_icon="🌍",
    layout="centered"
)

# ---------- CUSTOM CSS ----------
st.markdown("""
<style>

/* Background */
.stApp{
    background: linear-gradient(135deg,#74ebd5,#ACB6E5,#F6D365);
}

/* Title */
.title-box{
    background: linear-gradient(90deg,#ff512f,#dd2476);
    padding:20px;
    border-radius:15px;
    text-align:center;
    color:white;
    font-size:40px;
    font-weight:bold;
    margin-bottom:20px;
}

/* Score Box */
.score-box{
    background: linear-gradient(90deg,#11998e,#38ef7d);
    padding:15px;
    border-radius:15px;
    text-align:center;
    color:white;
    font-size:26px;
    font-weight:bold;
    margin-bottom:20px;
}

/* Input Header */
.input-box{
    background: linear-gradient(90deg,#36D1DC,#5B86E5);
    padding:15px;
    border-radius:15px;
    text-align:center;
    color:white;
    font-size:28px;
    font-weight:bold;
    margin-bottom:15px;
}

/* Result Box */
.result-box{
    background: linear-gradient(90deg,#FF512F,#F09819);
    padding:20px;
    border-radius:20px;
    text-align:center;
    color:white;
    font-size:30px;
    font-weight:bold;
}

/* Predict Button */
.stButton>button{
    width:100%;
    background: linear-gradient(90deg,#ff512f,#dd2476);
    color:white;
    font-size:20px;
    font-weight:bold;
    border:none;
    border-radius:12px;
    padding:10px;
}

.stButton>button:hover{
    background: linear-gradient(90deg,#11998e,#38ef7d);
    color:white;
}

</style>
""", unsafe_allow_html=True)


# ---------- MODEL ----------
@st.cache_resource
def load_and_train():

    data = pd.read_csv("cleaned_tourism_recommendation_dataset.csv")

    data["is_group_tour"] = data["is_group_tour"].map({"Yes":1,"No":0})

    numeric_cols = data.select_dtypes(include=["number"]).columns
    for col in numeric_cols:
        data[col] = data[col].fillna(data[col].median())

    categorical_cols = data.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        data[col] = data[col].fillna(data[col].mode()[0])

    X = data.drop("rating", axis=1)
    y = data["rating"]

    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    from sklearn.ensemble import RandomForestRegressor

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)

    score = model.score(X_test, y_test)

    return model, score


model, score = load_and_train()


# ---------- TITLE ----------
st.markdown("""
<div class="title-box">
🌍 Tourist Rating Prediction System
</div>
""", unsafe_allow_html=True)


# ---------- SCORE ----------
st.markdown(f"""
<div class="score-box">
📊 Model R² Score<br><br>
{score:.3f}
</div>
""", unsafe_allow_html=True)


# ---------- INPUT HEADER ----------
st.markdown("""
<div class="input-box">
🧳 Enter Tourist Details
</div>
""", unsafe_allow_html=True)


# ---------- INPUTS ----------
age = st.number_input("Age",1,100,25)

ticket_price = st.number_input(
    "Ticket Price",
    min_value=0.0,
    value=100.0
)

visit_duration_hours = st.number_input(
    "Visit Duration (Hours)",
    min_value=1.0,
    value=4.0
)

spend_amount = st.number_input(
    "Spend Amount",
    min_value=0.0,
    value=500.0
)

other_spend = st.number_input(
    "Other Spend",
    min_value=0.0,
    value=100.0
)

tour = st.selectbox(
    "Group Tour",
    ["No","Yes"]
)

group_fee = st.number_input(
    "Group Fee",
    min_value=0.0,
    value=0.0
)

trip_days = st.number_input(
    "Trip Days",
    min_value=1,
    max_value=30,
    value=2
)

group = 1 if tour=="Yes" else 0


# ---------- PREDICTION ----------
if st.button("Predict Tourist Rating"):

    input_data = pd.DataFrame(
        [[
            age,
            ticket_price,
            visit_duration_hours,
            spend_amount,
            other_spend,
            group,
            group_fee,
            trip_days
        ]],
        columns=[
            "age",
            "ticket_price",
            "visit_duration_hours",
            "spend_amount",
            "other_spend",
            "is_group_tour",
            "group_fee",
            "trip_days"
        ]
    )

    prediction = model.predict(input_data)

    st.balloons()

    st.markdown(f"""
    <div class="result-box">
    ⭐ Predicted Tourist Rating ⭐<br><br>
    {prediction[0]:.2f} / 5
    </div>
    """, unsafe_allow_html=True)

Overwriting app.py


In [35]:
from pyngrok import ngrok
import time

ngrok.kill()  # close any old tunnels first
ngrok.set_auth_token(ngrok_key)

public_url = ngrok.connect(8501)
print("Your app is live at:", public_url)

!streamlit run app.py &>/content/logs.txt &
time.sleep(4)
print("Streamlit backend running on Colab, frontend served through the ngrok link above.")

Your app is live at: NgrokTunnel: "https://pulsate-disloyal-wrongful.ngrok-free.dev" -> "http://localhost:8501"


Streamlit backend running on Colab, frontend served through the ngrok link above.


In [36]:
!rm -rf logs.txt && streamlit run app.py &>/content/logs.txt